<a id="top"></a>

# Lab L0606: parse the scratchpad contract

```yaml
title: "Lab L0606: parse the scratchpad contract"
keywords: scratchpad, thinking tags, answer tags, defensive parsing, fail loudly, regex, lab
estimated duration: 30
```

> **Lesson:** L06. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L06/objectives.md). Solutions: `L0606_lab_solutions.ipynb`. **Pure Python — no API key needed** (we parse crafted model replies, the same offline discipline as the L0206 lab).
>
> **You'll walk out able to** extract a final answer from a `<thinking>`/`<answer>` reply, fail loudly when the contract is broken, and explain why separating reasoning from answer matters.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Extract the answer, fail loudly](#2-problem-1--extract-the-answer-fail-loudly)
- [3. Problem 2 — Separate thinking from answer](#3-problem-2--separate-thinking-from-answer)
- [4. Problem 3 — Run it over every crafted case](#4-problem-3--run-it-over-every-crafted-case)
- [5. Problem 4 — Why separate thinking from answer? (written)](#5-problem-4--why-separate-thinking-from-answer-written)
- [6. Self-check](#6-self-check)

## 1. Setup

Five crafted replies, each bending the `<thinking>`/`<answer>` contract a different way. Real models are usually clean. Your parser still has to survive them when they're not.

In [1]:
import re

# Crafted model replies — each probes a different way the tag contract can bend.
REPLIES: dict[str, str] = {
    "clean": "<thinking>1 + 1 = 2, then 2 * 3 = 6.</thinking><answer>6</answer>",
    "prose_then_answer": "Let me work through it.\n<answer>6</answer>",
    "answer_outside": "<thinking>2 * 3 = 6</thinking>\nThe answer is 6.",
    "no_closing": "<thinking>2 * 3 = 6<answer>6</answer>",
    "no_tags_at_all": "I think it is 6.",
}
print(f"{len(REPLIES)} crafted replies ready")

5 crafted replies ready


[↑ Back to top](#top)

## 2. Problem 1 — Extract the answer, fail loudly

Implement `extract_answer`: return the contents of the first `<answer>...</answer>` block, stripped. If there is no well-formed `<answer>` block, **raise** `ValueError` with the raw reply — never silently return `""`. Use `re.DOTALL` so the block can span newlines.

In [2]:
def extract_answer(reply: str) -> str:
    """Return the <answer> contents, or raise loudly if the contract was broken."""
    match = re.search(r"<answer>(.*?)</answer>", reply, re.DOTALL)
    if match is None:
        raise ValueError(f"no well-formed <answer> block in reply:\n{reply!r}")
    return match.group(1).strip()


print(extract_answer(REPLIES["clean"]))  # expect: 6

6


[↑ Back to top](#top)

## 3. Problem 2 — Separate thinking from answer

Implement `split_reply` returning a `(thinking, answer)` tuple. `thinking` is the `<thinking>` contents or `None` if absent; `answer` reuses `extract_answer`. This is the split your downstream code wants: log `thinking`, consume `answer`.

In [3]:
def split_reply(reply: str) -> tuple[str | None, str]:
    """Return (thinking_or_None, answer). The answer is required; thinking is optional."""
    think_match = re.search(r"<thinking>(.*?)</thinking>", reply, re.DOTALL)
    thinking = think_match.group(1).strip() if think_match else None
    return thinking, extract_answer(reply)


print(split_reply(REPLIES["clean"]))  # expect: ("1 + 1 = 2, then 2 * 3 = 6.", "6")

('1 + 1 = 2, then 2 * 3 = 6.', '6')


[↑ Back to top](#top)

## 4. Problem 3 — Run it over every crafted case

Loop over `REPLIES`. For each, try `extract_answer` and print the answer; on `ValueError`, catch it and print a loud `CONTRACT BROKEN` notice and continue. Which replies parse, and which fail loudly?

In [4]:
for name, reply in REPLIES.items():
    try:
        answer = extract_answer(reply)
        print(f"{name:18} -> answer = {answer!r}")
    except ValueError:
        print(f"{name:18} -> CONTRACT BROKEN (fail loudly, do not guess)")

clean              -> answer = '6'
prose_then_answer  -> answer = '6'
answer_outside     -> CONTRACT BROKEN (fail loudly, do not guess)
no_closing         -> answer = '6'
no_tags_at_all     -> CONTRACT BROKEN (fail loudly, do not guess)


[↑ Back to top](#top)

## 5. Problem 4 — Why separate thinking from answer? (written)

In a sentence or two each, name **three** downstream consumers that benefit from the answer being in its own field rather than buried in prose. (Hint: think parsing, UX, and the evaluation/tracing lessons coming in L12–L13.)

_Write your answer by editing this cell (double-click)._

1. _TODO_
2. _TODO_
3. _TODO_

[↑ Back to top](#top)

## 6. Self-check

Compare against `L0606_lab_solutions.ipynb`. Done when `clean`, `prose_then_answer`, and `no_closing` parse to `6`, while `answer_outside` and `no_tags_at_all` **fail loudly**. Confirm your loop matches.

[↑ Back to top](#top)